### Sensor Location

In [1]:
import pandas as pd
import re
from pathlib import Path
from datetime import datetime

RAW_DIR = Path(".")  # notebook and CSV are in the same directory, use current directory
REPORT_PATH = Path("data_quality_report.md")  # Report is also generated in the same directory

FILES = {
    "sensor_location": RAW_DIR / "sensor_location_raw.csv",
    "pedestrian_hour_count": RAW_DIR / "pedestrian_hour_count_raw.csv",
    "landmark": RAW_DIR / "landmark_raw.csv",
    "pedestrian_minute_count": RAW_DIR / "minute_count_full_export.csv",
}

In [2]:
df_sensor = pd.read_csv(FILES["sensor_location"])
df_sensor.head()

,Location_ID,Sensor_Description,Sensor_Name,Installation_Date,Note,Location_Type,Status,Direction_1,Direction_2,Latitude,Longitude,Location
0,3,Melbourne Central,Swa295_T,2009-03-25,NaN,Outdoor,A,North,South,-37.811015,144.964295,"-37.81101524, 144.96429485"
1,5,Princes Bridge,PriNW_T,2009-03-26,Replace with: 00:6e:02:01:9e:54,Outdoor,A,North,South,-37.818742,144.967877,"-37.81874249, 144.96787656"
2,9,Southern Cross Station,Col700_T,2009-03-23,NaN,Outdoor,A,East,West,-37.819830,144.951026,"-37.81982992, 144.95102555"
3,11,Docklands Waterfront City Building Side,WatCit_T,2009-01-20,Device has been replaced with new sensor on 22...,Outdoor,A,East,West,-37.815667,144.939744,"-37.81566651, 144.93974366"
4,12,New Quay,NewQ_T,2009-01-21,NaN,Outdoor,A,East,West,-37.814580,144.942924,"-37.81457988, 144.94292398"


In [3]:
df_sensor.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 134 entries, 0 to 133
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Location_ID         134 non-null    int64  
 1   Sensor_Description  134 non-null    object 
 2   Sensor_Name         134 non-null    object 
 3   Installation_Date   133 non-null    object 
 4   Note                38 non-null     object 
 5   Location_Type       134 non-null    object 
 6   Status              134 non-null    object 
 7   Direction_1         100 non-null    object 
 8   Direction_2         100 non-null    object 
 9   Latitude            134 non-null    float64
 10  Longitude           134 non-null    float64
 11  Location            134 non-null    object 
dtypes: float64(2), int64(1), object(9)
memory usage: 12.7+ KB


In [4]:
df_sensor.isna().sum()

Location_ID            0
Sensor_Description     0
Sensor_Name            0
Installation_Date      1
Note                  96
Location_Type          0
Status                 0
Direction_1           34
Direction_2           34
Latitude               0
Longitude              0
Location               0
dtype: int64

In [5]:
df_sensor[df_sensor["Direction_1"].isna()]["Status"].value_counts()
df_sensor["Status"].value_counts()

Status
A    134
Name: count, dtype: int64

In [6]:
print("Location_ID duplicate count:", df_sensor["Location_ID"].duplicated().sum())

Location_ID duplicate count: 0


### Pedestrian hour count

In [7]:
df_hour = pd.read_csv(FILES["pedestrian_hour_count"], low_memory=False)
df_hour.head()

,ID,Location_ID,Sensing_Date,HourDay,Direction_1,Direction_2,Total_of_Directions,Sensor_Name,Location
0,49220260805,49,2026-08-05,2,6,11,17,Eli501_T,"-37.80730068, 144.95956055"
1,43020260805,43,2026-08-05,0,14,2,16,UM2_T,"-37.79844526, 144.96411782"
2,24120260805,24,2026-08-05,1,14,15,29,Col620_T,"-37.81887963, 144.95449198"
3,182120260805,182,2026-08-05,1,33,29,62,King163_T,"-37.81627451, 144.95550503"
4,30320260805,30,2026-08-05,3,0,1,1,Lon189_T,"-37.8112185, 144.96656806"


In [8]:
list(df_hour.columns)


['ID',
 'Location_ID',
 'Sensing_Date',
 'HourDay',
 'Direction_1',
 'Direction_2',
 'Total_of_Directions',
 'Sensor_Name',
 'Location']

In [9]:
df_hour.iloc[:, 3].unique()  # Column 4, i.e. the column after Sensing_Date

array([ 2,  0,  1,  3, 14, 15,  5,  4, 13, 17, 18, 21, 10, 12,  9, 19,  7,
       20, 11, 16,  8,  6, 22, 23])

In [10]:
df_hour["sensing_datetime"] = pd.to_datetime(df_hour["Sensing_Date"]) + pd.to_timedelta(df_hour["HourDay"], unit="h")
df_hour[["Sensing_Date", "HourDay", "sensing_datetime"]].head()

,Sensing_Date,HourDay,sensing_datetime
0,2026-08-05,2,2026-08-05 02:00:00
1,2026-08-05,0,2026-08-05 00:00:00
2,2026-08-05,1,2026-08-05 01:00:00
3,2026-08-05,1,2026-08-05 01:00:00
4,2026-08-05,3,2026-08-05 03:00:00


In [11]:
# Take the Location_ID from the first record, check what hours the hourly data has
sample_loc = df_hour.iloc[0]["Location_ID"]
sample_date = df_hour.iloc[0]["Sensing_Date"]
print("Location_ID:", sample_loc, "| Date:", sample_date, "| HourDay:", df_hour.iloc[0]["HourDay"])

# Find records for the same location_id and same day in the minute data, check which hour sensing_date/sensing_time falls in
df_minute = pd.read_csv(FILES["pedestrian_minute_count"], low_memory=False)
match = df_minute[(df_minute["location_id"] == sample_loc)]
match[["location_id", "sensing_date", "sensing_time", "sensing_datetime"]].head(10)

Location_ID: 49 | Date: 2026-08-05 | HourDay: 2


KeyError: 'location_id'

In [12]:
list(df_minute.columns)

['Location_ID',
 'Sensing_DateTime',
 'Sensing_Date',
 'Sensing_Time',
 'Direction_1',
 'Direction_2',
 'Total_of_Directions']

In [13]:
match = df_minute[df_minute["Location_ID"] == sample_loc]
match[["Location_ID", "Sensing_Date", "Sensing_Time", "Sensing_DateTime"]].head(10)

,Location_ID,Sensing_Date,Sensing_Time,Sensing_DateTime
304,49,2026-08-06,22:55,2026-08-06T22:55:00+10:00
448,49,2026-08-06,22:50,2026-08-06T22:50:00+10:00
651,49,2026-08-06,22:45,2026-08-06T22:45:00+10:00
893,49,2026-08-06,22:40,2026-08-06T22:40:00+10:00
1076,49,2026-08-06,22:35,2026-08-06T22:35:00+10:00
1318,49,2026-08-06,22:30,2026-08-06T22:30:00+10:00
1530,49,2026-08-06,22:25,2026-08-06T22:25:00+10:00
1717,49,2026-08-06,22:20,2026-08-06T22:20:00+10:00
1928,49,2026-08-06,22:15,2026-08-06T22:15:00+10:00
2175,49,2026-08-06,22:10,2026-08-06T22:10:00+10:00


In [14]:
match = df_minute[
    (df_minute["Location_ID"] == sample_loc) &
    (df_minute["Sensing_Date"] == sample_date)
]
match_hour2 = match[match["Sensing_Time"].str.startswith("02:")]
match_hour2[["Location_ID", "Sensing_Date", "Sensing_Time", "Sensing_DateTime", "Total_of_Directions"]]

,Location_ID,Sensing_Date,Sensing_Time,Sensing_DateTime,Total_of_Directions
109417,49,2026-08-05,02:55,2026-08-05T02:55:00+10:00,2
109459,49,2026-08-05,02:50,2026-08-05T02:50:00+10:00,1
109514,49,2026-08-05,02:45,2026-08-05T02:45:00+10:00,1
109643,49,2026-08-05,02:30,2026-08-05T02:30:00+10:00,4
109698,49,2026-08-05,02:25,2026-08-05T02:25:00+10:00,1
109784,49,2026-08-05,02:15,2026-08-05T02:15:00+10:00,1
109815,49,2026-08-05,02:10,2026-08-05T02:10:00+10:00,4
109876,49,2026-08-05,02:05,2026-08-05T02:05:00+10:00,2
109912,49,2026-08-05,02:00,2026-08-05T02:00:00+10:00,1


In [15]:
match_hour2["Total_of_Directions"].sum()

np.int64(17)

In [16]:
df_hour.info()
df_hour.isna().sum()
print("Fully duplicate rows:", df_hour.duplicated().sum())
print("ID column duplicates:", df_hour["ID"].duplicated().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1613800 entries, 0 to 1613799
Data columns (total 10 columns):
 #   Column               Non-Null Count    Dtype         
---  ------               --------------    -----         
 0   ID                   1613800 non-null  int64         
 1   Location_ID          1613800 non-null  int64         
 2   Sensing_Date         1613800 non-null  object        
 3   HourDay              1613800 non-null  int64         
 4   Direction_1          1613800 non-null  int64         
 5   Direction_2          1613800 non-null  int64         
 6   Total_of_Directions  1613800 non-null  int64         
 7   Sensor_Name          1588298 non-null  object        
 8   Location             1588298 non-null  object        
 9   sensing_datetime     1613800 non-null  datetime64[ns]
dtypes: datetime64[ns](1), int64(6), object(3)
memory usage: 123.1+ MB
Fully duplicate rows: 0
ID column duplicates: 66343


In [17]:
# Find a few duplicate IDs, check whether their Location_ID/HourDay/Sensing_Date are actually different
dup_ids = df_hour[df_hour["ID"].duplicated(keep=False)]
dup_ids.sort_values("ID").head(10)[["ID", "Location_ID", "HourDay", "Sensing_Date"]]

,ID,Location_ID,HourDay,Sensing_Date
1613493,11020240806,11,0,2024-08-06
1613352,11020240806,1,10,2024-08-06
1610221,11020240807,11,0,2024-08-07
1610960,11020240807,1,10,2024-08-07
1609479,11020240808,1,10,2024-08-08
1608204,11020240808,11,0,2024-08-08
1607194,11020240809,11,0,2024-08-09
1607983,11020240809,1,10,2024-08-09
1605162,11020240810,1,10,2024-08-10
1605656,11020240810,11,0,2024-08-10


In [18]:
composite_dup = df_hour.duplicated(subset=["Location_ID", "Sensing_Date", "HourDay"]).sum()
print("Composite key (Location_ID+Date+Hour) duplicate count:", composite_dup)

Composite key (Location_ID+Date+Hour) duplicate count: 0


### minute_count_full_export

In [19]:
df_minute.info()
df_minute.isna().sum() / len(df_minute) * 100
print("Fully duplicate rows:", df_minute.duplicated().sum())
print("Composite key (Location_ID+Sensing_DateTime) duplicate count:", 
      df_minute.duplicated(subset=["Location_ID", "Sensing_DateTime"]).sum())
for col in ["Direction_1", "Direction_2", "Total_of_Directions"]:
    print(f"{col}: negative {(df_minute[col] < 0).sum()} rows | max {df_minute[col].max()} | min {df_minute[col].min()}")
mismatch = (df_minute["Direction_1"] + df_minute["Direction_2"] != df_minute["Total_of_Directions"]).sum()
print("Records where direction sum does not match Total:", mismatch)
print("Number of Location_IDs covered:", df_minute["Location_ID"].nunique())
print("Total sensors in Sensor Locations table:", df_sensor["Location_ID"].nunique())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 221543 entries, 0 to 221542
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   Location_ID          221543 non-null  int64 
 1   Sensing_DateTime     221543 non-null  object
 2   Sensing_Date         221543 non-null  object
 3   Sensing_Time         221543 non-null  object
 4   Direction_1          221543 non-null  int64 
 5   Direction_2          221543 non-null  int64 
 6   Total_of_Directions  221543 non-null  int64 
dtypes: int64(4), object(3)
memory usage: 11.8+ MB
Fully duplicate rows: 0
Composite key (Location_ID+Sensing_DateTime) duplicate count: 90
Direction_1: negative 0 rows | max 297 | min 0
Direction_2: negative 0 rows | max 330 | min 0
Total_of_Directions: negative 0 rows | max 533 | min 0
Records where direction sum does not match Total: 0
Number of Location_IDs covered: 99
Total sensors in Sensor Locations table: 134


In [20]:
dup_rows = df_minute[df_minute.duplicated(subset=["Location_ID", "Sensing_DateTime"], keep=False)]
dup_rows["Location_ID"].value_counts()

Location_ID
11     78
35     58
59      8
132     8
46      8
161     8
123     6
43      2
185     2
47      2
Name: count, dtype: int64

In [21]:
sensor_ids_all = set(df_sensor["Location_ID"])
sensor_ids_in_minute = set(df_minute["Location_ID"].unique())
missing_ids = sensor_ids_all - sensor_ids_in_minute
print("Missing Location_IDs:", sorted(missing_ids))
print("Missing count:", len(missing_ids))

Missing Location_IDs: [80, 81, 82, 83, 89, 90, 91, 92, 93, 94, 95, 96, 99, 102, 103, 104, 105, 106, 108, 116, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 158, 159, 160]
Missing count: 35


In [22]:
sample_dup = df_minute[df_minute["Location_ID"] == 11].sort_values("Sensing_DateTime")
sample_dup[sample_dup.duplicated(subset=["Sensing_DateTime"], keep=False)].head(10)

,Location_ID,Sensing_DateTime,Sensing_Date,Sensing_Time,Direction_1,Direction_2,Total_of_Directions
204511,11,2026-08-03T10:30:00+10:00,2026-08-03,10:30,0,2,2
204494,11,2026-08-03T10:30:00+10:00,2026-08-03,10:30,1,5,6
199683,11,2026-08-03T12:00:00+10:00,2026-08-03,12:00,2,1,3
199637,11,2026-08-03T12:00:00+10:00,2026-08-03,12:00,6,3,9
196334,11,2026-08-03T13:00:00+10:00,2026-08-03,13:00,2,5,7
196358,11,2026-08-03T13:00:00+10:00,2026-08-03,13:00,6,7,13
194674,11,2026-08-03T13:30:00+10:00,2026-08-03,13:30,4,1,5
194649,11,2026-08-03T13:30:00+10:00,2026-08-03,13:30,3,0,3
193024,11,2026-08-03T14:00:00+10:00,2026-08-03,14:00,2,0,2
193074,11,2026-08-03T14:00:00+10:00,2026-08-03,14:00,4,1,5


In [23]:
missing_ids = [80, 81, 82, 83, 89, 90, 91, 92, 93, 94, 95, 96, 99, 102, 103, 104, 105, 106, 108, 116, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 158, 159, 160]
df_sensor[df_sensor["Location_ID"].isin(missing_ids)][["Location_ID", "Sensor_Description", "Installation_Date", "Note", "Status"]]

,Location_ID,Sensor_Description,Installation_Date,Note,Status
26,81,Boyd Commuinty - Front door,2021-03-30,NaN,A
27,82,512 Elizabeth Street,2021-11-05,NaN,A
30,89,City Library,2013-12-11,NaN,A
31,90,Boyd Community Hub- Library,2015-08-11,NaN,A
32,93,East Melbourne Library,2013-02-02,NaN,A
33,96,Fitzroy Garden Visitor Centre Internal,2015-03-02,"Pushbox Upgrade, 30/06/2023",A
34,105,Kathleen Syme Library Lib,2018-02-02,NaN,A
35,108,William St - Little Lonsdale St (West),2022-10-08,Added lines for 7 Eleven Store - 2025-05-07,A
36,116,Fitzroy Garden Visitor Centre Cafe Verandah,2015-03-02,"Pushbox Upgrade, 30/06/2023",A
45,146,narrm ngarrgu Library - Level 1 - Lift 3,2023-10-23,NaN,A


In [24]:
missing_df = df_sensor[df_sensor["Location_ID"].isin(missing_ids)]
missing_df["Location_Type"].value_counts()

all_df_type = df_sensor["Location_Type"].value_counts()
print(all_df_type)

Location_Type
Outdoor    100
Indoor      34
Name: count, dtype: int64


In [25]:
indoor_ids = set(df_sensor[df_sensor["Location_Type"] == "Indoor"]["Location_ID"])
direction_missing_ids = set(df_sensor[df_sensor["Direction_1"].isna()]["Location_ID"])
print("Fully matches:", indoor_ids == direction_missing_ids)
print("Difference:", indoor_ids.symmetric_difference(direction_missing_ids))

Fully matches: True
Difference: set()


In [26]:
missing_df = df_sensor[df_sensor["Location_ID"].isin(missing_ids)]
print(missing_df["Location_Type"].value_counts())

# Find out whether that "extra 1" is Indoor or Outdoor
extra = set(missing_ids) - indoor_ids
print("IDs missing minute data but not Indoor:", extra)
if extra:
    print(df_sensor[df_sensor["Location_ID"].isin(extra)])

Location_Type
Indoor     34
Outdoor     1
Name: count, dtype: int64
IDs missing minute data but not Indoor: {108}
    Location_ID                      Sensor_Description Sensor_Name  \
35          108  William St - Little Lonsdale St (West)   261Will_T   

   Installation_Date                                         Note  \
35        2022-10-08  Added lines for 7 Eleven Store - 2025-05-07   

   Location_Type Status Direction_1 Direction_2   Latitude   Longitude  \
35       Outdoor      A       North       South -37.812958  144.956788   

                      Location  
35  -37.81295822, 144.95678789  


### Landmark

In [27]:
df_landmark = pd.read_csv(FILES["landmark"], low_memory=False)
df_landmark.head()
df_landmark.info()
df_landmark.isna().sum() / len(df_landmark) * 100
print("Fully duplicate rows:", df_landmark.duplicated().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 242 entries, 0 to 241
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Theme         242 non-null    object
 1   Sub Theme     242 non-null    object
 2   Feature Name  242 non-null    object
 3   Co-ordinates  242 non-null    object
dtypes: object(4)
memory usage: 7.7+ KB
Fully duplicate rows: 0


In [28]:
df_landmark.head(10)
print("Feature Name duplicate count:", df_landmark["Feature Name"].duplicated().sum())

# Check the actual format of Co-ordinates, to know how to split it
df_landmark["Co-ordinates"].head(5).tolist()

# Theme/Sub Theme spelling consistency check
print("Theme unique values:", sorted(df_landmark["Theme"].unique()))
print()
print("Sub Theme unique values:", sorted(df_landmark["Sub Theme"].unique()))

Feature Name duplicate count: 9
Theme unique values: ['Community Use', 'Education Centre', 'Health Services', 'Industrial', 'Leisure/Recreation', 'Mixed Use', 'Office', 'Place Of Assembly', 'Place of Worship', 'Purpose Built', 'Residential Accommodation', 'Retail', 'Specialist Residential Accommodation', 'Transport', 'Vacant Land', 'Warehouse/Store']

Sub Theme unique values: ['Aquarium', 'Art Gallery/Museum', 'Bridge', 'Casino', 'Cemetery', 'Church', 'Cinema', 'Current Construction Site', 'Current Construction Site - Commercial', 'Department Store', 'Dwelling (House)', 'Film & RV Studio', 'Fire Station', 'Function/Conference/Exhibition Centre', 'Further Education', 'Government Building', 'Gymnasium/Health Club', 'Hostel', 'Indoor Recreation Facility', 'Industrial (Manufacturing)', 'Informal Outdoor Facility (Park/Garden/Reserve)', 'Library', 'Major Sports & Recreation Facility', 'Marina', 'Medical Services', 'Observation Tower/Wheel', 'Office', 'Outdoor Recreation Facility (Zoo, Golf 

In [29]:
dup_names = df_landmark[df_landmark["Feature Name"].duplicated(keep=False)].sort_values("Feature Name")
dup_names

,Theme,Sub Theme,Feature Name,Co-ordinates
25,Transport,Railway Station,Flagstaff Railway Station,"-37.8122356514626, 144.956318211113"
119,Transport,Railway Station,Flagstaff Railway Station,"-37.8116384403974, 144.9561186805"
5,Transport,Railway Station,Melbourne Central Railway Station,"-37.8100167030855, 144.963789344893"
204,Transport,Railway Station,Melbourne Central Railway Station,"-37.8097759325888, 144.962670059539"
219,Transport,Railway Station,Melbourne Central Railway Station,"-37.8105992402369, 144.96169061383"
220,Transport,Railway Station,Melbourne Central Railway Station,"-37.8108930047401, 144.963100728702"
1,Transport,Railway Station,Parliament Railway Station,"-37.8116061787171, 144.973017263156"
92,Transport,Railway Station,Parliament Railway Station,"-37.8127621113552, 144.973433796529"
132,Transport,Railway Station,Parliament Railway Station,"-37.8095762219384, 144.972330132127"
233,Transport,Railway Station,Parliament Railway Station,"-37.8121572789173, 144.973742010011"


## Step3

In [30]:
candidates_keywords = ["Flinders", "Melbourne Central", "Southern Cross", "Collins", "Princes Bridge", "QV", "Bourke", "Swanston"]

for kw in candidates_keywords:
    matches = df_sensor[df_sensor["Sensor_Description"].str.contains(kw, case=False, na=False)]
    if len(matches) > 0:
        print(f"=== Keyword: {kw} ===")
        print(matches[["Location_ID", "Sensor_Description", "Location_Type", "Status"]])
        print()
        

=== Keyword: Flinders ===
     Location_ID                                 Sensor_Description  \
11            41                     Flinders La-Swanston St (West)   
23            68                   Flinders Ln -Degraves St (North)   
24            69                Flinders Ln -Degraves St (Crossing)   
25            72                                  Flinders St- ACMI   
37           117              114 Flinders Street Car Park Footpath   
55             6                 Flinders Underpass - Myki Barriers   
69            75                      Spring St- Flinders st (West)   
71            79                                Flinders St (South)   
74            84   Elizabeth St - Flinders St (East) - New footpath   
87           141   Awning of Nationwide Parking 474 Flinders Street   
97           209                       Flinders Underpass - Walkway   
109           67                   Flinders Ln -Degraves St (South)   
117          118              114 Flinders Street C

In [31]:
selected_ids = [5, 3, 9, 27, 1, 53]

print("Hourly data coverage:")
print(df_hour[df_hour["Location_ID"].isin(selected_ids)]["Location_ID"].value_counts())

print("\nMinute data coverage:")
print(df_minute[df_minute["Location_ID"].isin(selected_ids)]["Location_ID"].value_counts())

Hourly data coverage:
Location_ID
3     17498
5     17486
9     17451
27    17427
53    17111
1     11689
Name: count, dtype: int64

Minute data coverage:
Location_ID
3     4866
53    4253
1     4231
9     4105
5     1111
27     950
Name: count, dtype: int64


In [32]:
df_sensor[df_sensor["Location_ID"].isin([5,3,9,27,1,53])][["Location_ID","Sensor_Description","Installation_Date"]]

,Location_ID,Sensor_Description,Installation_Date
0,3,Melbourne Central,2009-03-25
1,5,Princes Bridge,2009-03-26
2,9,Southern Cross Station,2009-03-23
9,27,QV Market-Peel St,2013-08-16
20,53,Collins Street (North),2015-09-23
53,1,Bourke Street Mall (North),2009-03-24


In [33]:
for loc_id in [5, 27]:
    subset = df_minute[df_minute["Location_ID"] == loc_id]
    print(f"Location_ID={loc_id}:")
    print(subset["Sensing_Date"].value_counts().sort_index())
    print()

Location_ID=5:
Sensing_Date
2026-08-02      1
2026-08-03    280
2026-08-04    282
2026-08-05    279
2026-08-06    269
Name: count, dtype: int64

Location_ID=27:
Sensing_Date
2026-08-02      1
2026-08-03    238
2026-08-04    237
2026-08-05    240
2026-08-06    234
Name: count, dtype: int64



In [36]:
df_hour[df_hour["Location_ID"] == 1]["Sensing_Date"].agg(["min", "max"])
# Compare the date ranges of the other locations, check whether Bourke St Mall inherently has a shorter range
for loc_id in [5, 3, 9, 27, 1, 53]:
    dates = df_hour[df_hour["Location_ID"] == loc_id]["Sensing_Date"]
    print(f"Location_ID={loc_id}: {dates.min()} to {dates.max()}, {dates.nunique()} days total")

Location_ID=5: 2024-08-06 to 2026-08-05, 730 days total
Location_ID=3: 2024-08-06 to 2026-08-05, 730 days total
Location_ID=9: 2024-08-06 to 2026-08-05, 730 days total
Location_ID=27: 2024-08-06 to 2026-08-05, 730 days total
Location_ID=1: 2024-08-06 to 2026-08-05, 491 days total
Location_ID=53: 2024-08-06 to 2026-08-05, 716 days total


In [37]:
import pandas as pd

full_range = pd.date_range(start="2024-08-06", end="2026-08-05")
existing_dates = pd.to_datetime(df_hour[df_hour["Location_ID"] == 1]["Sensing_Date"].unique())
missing_dates = full_range.difference(existing_dates)

print(f"Total missing: {len(missing_dates)} days")
print("Any missing in the last 30 days:", missing_dates[missing_dates >= (pd.Timestamp.now() - pd.Timedelta(days=30))].tolist())
print("\nMissing date examples (first 20):")
print(missing_dates[:20].tolist())

Total missing: 239 days
Any missing in the last 30 days: []

Missing date examples (first 20):
[Timestamp('2024-09-03 00:00:00'), Timestamp('2024-09-04 00:00:00'), Timestamp('2024-09-05 00:00:00'), Timestamp('2024-09-06 00:00:00'), Timestamp('2024-09-07 00:00:00'), Timestamp('2024-09-08 00:00:00'), Timestamp('2024-09-09 00:00:00'), Timestamp('2024-09-10 00:00:00'), Timestamp('2025-11-11 00:00:00'), Timestamp('2025-11-12 00:00:00'), Timestamp('2025-11-13 00:00:00'), Timestamp('2025-11-14 00:00:00'), Timestamp('2025-11-15 00:00:00'), Timestamp('2025-11-16 00:00:00'), Timestamp('2025-11-17 00:00:00'), Timestamp('2025-11-18 00:00:00'), Timestamp('2025-11-19 00:00:00'), Timestamp('2025-11-20 00:00:00'), Timestamp('2025-11-21 00:00:00'), Timestamp('2025-11-22 00:00:00')]


## Step4

In [38]:
import pandas as pd

# Use the latest date in the hourly data as the window end date
latest_date = pd.to_datetime(df_hour["Sensing_Date"]).max()
window_start = latest_date - pd.Timedelta(weeks=8) + pd.Timedelta(days=1)

print(f"Candidate window: {window_start.date()} to {latest_date.date()} (56 days total)")

full_window = pd.date_range(start=window_start, end=latest_date)
selected_ids = [5, 3, 9, 27, 1, 53]

print("\nMissing data per location within this window:")
for loc_id in selected_ids:
    existing = pd.to_datetime(df_hour[df_hour["Location_ID"] == loc_id]["Sensing_Date"].unique())
    missing_in_window = full_window.difference(existing)
    print(f"Location_ID={loc_id}: missing {len(missing_in_window)} days", 
          missing_in_window.tolist() if len(missing_in_window) > 0 else "")

Candidate window: 2026-06-11 to 2026-08-05 (56 days total)

Missing data per location within this window:
Location_ID=5: missing 0 days 
Location_ID=3: missing 0 days 
Location_ID=9: missing 0 days 
Location_ID=27: missing 0 days 
Location_ID=1: missing 19 days [Timestamp('2026-06-11 00:00:00'), Timestamp('2026-06-12 00:00:00'), Timestamp('2026-06-13 00:00:00'), Timestamp('2026-06-14 00:00:00'), Timestamp('2026-06-15 00:00:00'), Timestamp('2026-06-16 00:00:00'), Timestamp('2026-06-17 00:00:00'), Timestamp('2026-06-18 00:00:00'), Timestamp('2026-06-19 00:00:00'), Timestamp('2026-06-20 00:00:00'), Timestamp('2026-06-21 00:00:00'), Timestamp('2026-06-22 00:00:00'), Timestamp('2026-06-23 00:00:00'), Timestamp('2026-06-24 00:00:00'), Timestamp('2026-06-25 00:00:00'), Timestamp('2026-06-26 00:00:00'), Timestamp('2026-06-27 00:00:00'), Timestamp('2026-06-28 00:00:00'), Timestamp('2026-06-29 00:00:00')]
Location_ID=53: missing 0 days 


In [40]:
import pandas as pd

selected_ids = [5, 3, 9, 27, 1, 53]

# Collect the set of dates with data for each location
date_sets = {}
for loc_id in selected_ids:
    date_sets[loc_id] = set(pd.to_datetime(df_hour[df_hour["Location_ID"] == loc_id]["Sensing_Date"].unique()))

# Walk backward day by day from the latest date, to find the first window where "all 6 locations have zero missing days in a 56-day window"
latest_date = pd.to_datetime(df_hour["Sensing_Date"]).max()
earliest_date = pd.to_datetime(df_hour["Sensing_Date"]).min()

window_end = latest_date
found = False

while window_end - pd.Timedelta(weeks=8) + pd.Timedelta(days=1) >= earliest_date:
    window_start = window_end - pd.Timedelta(weeks=8) + pd.Timedelta(days=1)
    full_window = set(pd.date_range(start=window_start, end=window_end))
    
    all_clean = True
    for loc_id in selected_ids:
        missing = full_window - date_sets[loc_id]
        if len(missing) > 0:
            all_clean = False
            break
    
    if all_clean:
        print(f"Found zero-missing window: {window_start.date()} to {window_end.date()}")
        found = True
        break
    
    window_end -= pd.Timedelta(days=1)

if not found:
    print("No 56-day window with zero missing days across all 6 locations was found within the full 730-day range")

Found zero-missing window: 2025-09-16 to 2025-11-10


In [42]:
window_start = pd.Timestamp("2025-09-16")
window_end = pd.Timestamp("2025-11-10")

# 1. Re-confirm that all 24 hours are present for every day across these 56 days for every location (not just "has a record that day", but truly complete for every hour)
for loc_id in selected_ids:
    subset = df_hour[
        (df_hour["Location_ID"] == loc_id) &
        (pd.to_datetime(df_hour["Sensing_Date"]) >= window_start) &
        (pd.to_datetime(df_hour["Sensing_Date"]) <= window_end)
    ]
    expected_rows = 56 * 24  # 56 days x 24 hours per day
    actual_rows = len(subset)
    print(f"Location_ID={loc_id}: expected {expected_rows} rows, actual {actual_rows} rows, {'complete' if actual_rows == expected_rows else 'has gaps'}")

Location_ID=5: expected 1344 rows, actual 1341 rows, has gaps
Location_ID=3: expected 1344 rows, actual 1343 rows, has gaps
Location_ID=9: expected 1344 rows, actual 1339 rows, has gaps
Location_ID=27: expected 1344 rows, actual 1341 rows, has gaps
Location_ID=1: expected 1344 rows, actual 1326 rows, has gaps
Location_ID=53: expected 1344 rows, actual 1343 rows, has gaps


In [43]:
for loc_id in [1]:  # Focus on Bourke St Mall, which has the largest gap
    subset = df_hour[
        (df_hour["Location_ID"] == loc_id) &
        (pd.to_datetime(df_hour["Sensing_Date"]) >= window_start) &
        (pd.to_datetime(df_hour["Sensing_Date"]) <= window_end)
    ]
    full_hours = pd.MultiIndex.from_product(
        [pd.date_range(window_start, window_end), range(24)],
        names=["date", "hour"]
    )
    existing_hours = pd.MultiIndex.from_arrays(
        [pd.to_datetime(subset["Sensing_Date"]), subset["HourDay"]]
    )
    missing_hours = full_hours.difference(existing_hours)
    print(missing_hours.tolist())

[(Timestamp('2025-10-05 00:00:00'), 2), (Timestamp('2025-11-10 00:00:00'), 7), (Timestamp('2025-11-10 00:00:00'), 8), (Timestamp('2025-11-10 00:00:00'), 9), (Timestamp('2025-11-10 00:00:00'), 10), (Timestamp('2025-11-10 00:00:00'), 11), (Timestamp('2025-11-10 00:00:00'), 12), (Timestamp('2025-11-10 00:00:00'), 13), (Timestamp('2025-11-10 00:00:00'), 14), (Timestamp('2025-11-10 00:00:00'), 15), (Timestamp('2025-11-10 00:00:00'), 16), (Timestamp('2025-11-10 00:00:00'), 17), (Timestamp('2025-11-10 00:00:00'), 18), (Timestamp('2025-11-10 00:00:00'), 19), (Timestamp('2025-11-10 00:00:00'), 20), (Timestamp('2025-11-10 00:00:00'), 21), (Timestamp('2025-11-10 00:00:00'), 22), (Timestamp('2025-11-10 00:00:00'), 23)]


In [44]:
window_start_v2 = pd.Timestamp("2025-09-15")
window_end_v2 = pd.Timestamp("2025-11-09")  # Shift back 1 day to avoid the outage on that day

for loc_id in selected_ids:
    subset = df_hour[
        (df_hour["Location_ID"] == loc_id) &
        (pd.to_datetime(df_hour["Sensing_Date"]) >= window_start_v2) &
        (pd.to_datetime(df_hour["Sensing_Date"]) <= window_end_v2)
    ]
    expected_rows = 56 * 24
    actual_rows = len(subset)
    print(f"Location_ID={loc_id}: expected {expected_rows} rows, actual {actual_rows} rows, {'complete' if actual_rows == expected_rows else f'missing {expected_rows-actual_rows} hours'}")

Location_ID=5: expected 1344 rows, actual 1341 rows, missing 3 hours
Location_ID=3: expected 1344 rows, actual 1343 rows, missing 1 hours
Location_ID=9: expected 1344 rows, actual 1339 rows, missing 5 hours
Location_ID=27: expected 1344 rows, actual 1341 rows, missing 3 hours
Location_ID=1: expected 1344 rows, actual 1343 rows, missing 1 hours
Location_ID=53: expected 1344 rows, actual 1343 rows, missing 1 hours


## Step 5 Clean sensor_location

In [45]:
selected_ids = [5, 3, 9, 27, 1, 53]

df_sensor_clean = df_sensor[df_sensor["Location_ID"].isin(selected_ids)].copy()

# Drop the redundant Location column (latitude/longitude already have independent columns)
df_sensor_clean = df_sensor_clean.drop(columns=["Location"])

# Standardize column names to lowercase with underscores, for easier database building later
df_sensor_clean.columns = [c.lower() for c in df_sensor_clean.columns]

print(df_sensor_clean.shape)
df_sensor_clean

(6, 11)


,location_id,sensor_description,sensor_name,installation_date,note,location_type,status,direction_1,direction_2,latitude,longitude
0,3,Melbourne Central,Swa295_T,2009-03-25,NaN,Outdoor,A,North,South,-37.811015,144.964295
1,5,Princes Bridge,PriNW_T,2009-03-26,Replace with: 00:6e:02:01:9e:54,Outdoor,A,North,South,-37.818742,144.967877
2,9,Southern Cross Station,Col700_T,2009-03-23,NaN,Outdoor,A,East,West,-37.819830,144.951026
9,27,QV Market-Peel St,Vic_T,2013-08-16,NaN,Outdoor,A,East,West,-37.806069,144.956447
20,53,Collins Street (North),Col254_T,2015-09-23,NaN,Outdoor,A,East,West,-37.815642,144.965499
53,1,Bourke Street Mall (North),Bou292_T,2009-03-24,NaN,Outdoor,A,East,West,-37.813494,144.965153


In [46]:
import os
os.makedirs("data/cleaned", exist_ok=True)
df_sensor_clean.to_csv("data/cleaned/sensor_location.csv", index=False)
print("Saved:", len(df_sensor_clean), "rows")

Saved: 6 rows


## Step 6 Clean pedestrian_hour_count

In [47]:
window_start = pd.Timestamp("2025-09-15")
window_end = pd.Timestamp("2025-11-09")

df_hour_clean = df_hour[
    (df_hour["Location_ID"].isin(selected_ids)) &
    (pd.to_datetime(df_hour["Sensing_Date"]) >= window_start) &
    (pd.to_datetime(df_hour["Sensing_Date"]) <= window_end)
].copy()

# Drop the flawed original ID and redundant columns
df_hour_clean = df_hour_clean.drop(columns=["ID", "Location", "Sensor_Name"])

# Rename HourDay to the clearer hour_of_day
df_hour_clean = df_hour_clean.rename(columns={"HourDay": "hour_of_day"})

# Construct the standard timestamp (already verified to be local time, Australia/Melbourne)
df_hour_clean["sensing_datetime"] = pd.to_datetime(df_hour_clean["Sensing_Date"]) + pd.to_timedelta(df_hour_clean["hour_of_day"], unit="h")
df_hour_clean["sensing_datetime"] = df_hour_clean["sensing_datetime"].dt.tz_localize("Australia/Melbourne")

# Standardize column names to lowercase
df_hour_clean.columns = [c.lower() for c in df_hour_clean.columns]

print(df_hour_clean.shape)
df_hour_clean.head()

(8050, 7)


,location_id,sensing_date,hour_of_day,direction_1,direction_2,total_of_directions,sensing_datetime
603759,9,2025-11-09,12,57,85,142,2025-11-09 12:00:00+11:00
603782,27,2025-11-09,22,17,33,50,2025-11-09 22:00:00+11:00
603820,5,2025-11-09,0,200,170,370,2025-11-09 00:00:00+11:00
603841,27,2025-11-09,8,59,35,94,2025-11-09 08:00:00+11:00
603844,3,2025-11-09,19,874,925,1799,2025-11-09 19:00:00+11:00


In [48]:
df_hour_clean.to_csv("data/cleaned/pedestrian_hour_count.csv", index=False)
print("Saved:", len(df_hour_clean), "rows")

Saved: 8050 rows


## Step 7 Clean pedestrian_minute_count

In [49]:
df_minute_clean = df_minute[df_minute["Location_ID"].isin(selected_ids)].copy()

# Handle duplicate records at the same timestamp: keep the one with the larger Total_of_Directions (considered the more complete reading)
df_minute_clean = df_minute_clean.sort_values("Total_of_Directions", ascending=False)
df_minute_clean = df_minute_clean.drop_duplicates(subset=["Location_ID", "Sensing_DateTime"], keep="first")

# Standardize column names to lowercase
df_minute_clean.columns = [c.lower() for c in df_minute_clean.columns]

print(df_minute_clean.shape)
df_minute_clean.head()

(19516, 7)


,location_id,sensing_datetime,sensing_date,sensing_time,direction_1,direction_2,total_of_directions
3364,5,2026-08-06T21:40:00+10:00,2026-08-06,21:40,289,41,330
76130,5,2026-08-05T16:15:00+10:00,2026-08-05,16:15,284,44,328
3151,5,2026-08-06T21:45:00+10:00,2026-08-06,21:45,295,29,324
75821,5,2026-08-05T16:20:00+10:00,2026-08-05,16:20,195,50,245
16061,5,2026-08-06T17:20:00+10:00,2026-08-06,17:20,133,92,225


In [50]:
df_minute_clean.to_csv("data/cleaned/pedestrian_minute_count.csv", index=False)
print("Saved:", len(df_minute_clean), "rows")

Saved: 19516 rows


## Step 8 Clean Landmark

In [51]:
target_sub_themes = ["Library", "Informal Outdoor Facility (Park/Garden/Reserve)"]

df_landmark_clean = df_landmark[df_landmark["Sub Theme"].isin(target_sub_themes)].copy()

# Split Co-ordinates into separate latitude/longitude columns
coords_split = df_landmark_clean["Co-ordinates"].str.split(",", expand=True)
df_landmark_clean["latitude"] = coords_split[0].astype(float)
df_landmark_clean["longitude"] = coords_split[1].astype(float)
df_landmark_clean = df_landmark_clean.drop(columns=["Co-ordinates"])

# Standardize column names
df_landmark_clean.columns = [c.lower().replace(" ", "_") for c in df_landmark_clean.columns]

print(df_landmark_clean.shape)
df_landmark_clean.head(10)

(38, 5)


,theme,sub_theme,feature_name,latitude,longitude
2,Leisure/Recreation,Informal Outdoor Facility (Park/Garden/Reserve),North Melbourne Recreation Reserve,-37.798835,144.941452
3,Leisure/Recreation,Informal Outdoor Facility (Park/Garden/Reserve),Princes Park,-37.787016,144.961115
6,Leisure/Recreation,Informal Outdoor Facility (Park/Garden/Reserve),Argyle Square,-37.803148,144.965761
12,Leisure/Recreation,Informal Outdoor Facility (Park/Garden/Reserve),Kings Domain,-37.825524,144.974108
17,Leisure/Recreation,Informal Outdoor Facility (Park/Garden/Reserve),Federation Square,-37.817852,144.968964
21,Leisure/Recreation,Informal Outdoor Facility (Park/Garden/Reserve),Flagstaff Gardens,-37.811122,144.954696
23,Leisure/Recreation,Informal Outdoor Facility (Park/Garden/Reserve),Westgate Park,-37.831492,144.908825
26,Leisure/Recreation,Informal Outdoor Facility (Park/Garden/Reserve),Newmarket Reserve,-37.787847,144.922972
27,Leisure/Recreation,Informal Outdoor Facility (Park/Garden/Reserve),J.J Holland Park,-37.798236,144.923837
30,Leisure/Recreation,Informal Outdoor Facility (Park/Garden/Reserve),Fitzroy Gardens,-37.812962,144.980456


In [52]:
# Generate the landmark_category table: unique category combinations, assign an auto-increment ID
df_landmark_category = df_landmark_clean[["theme", "sub_theme"]].drop_duplicates().reset_index(drop=True)
df_landmark_category["category_id"] = df_landmark_category.index + 1

print("landmark_category:")
print(df_landmark_category)

# Generate the landmark table: link to category_id, no longer duplicate the theme/sub_theme text
df_landmark_final = df_landmark_clean.merge(df_landmark_category, on=["theme", "sub_theme"])
df_landmark_final = df_landmark_final.drop(columns=["theme", "sub_theme"])
df_landmark_final = df_landmark_final.reset_index(drop=True)
df_landmark_final["landmark_id"] = df_landmark_final.index + 1  # Auto-increment primary key, replacing the unreliable feature_name

# Adjust column order
df_landmark_final = df_landmark_final[["landmark_id", "feature_name", "category_id", "latitude", "longitude"]]

print("\nlandmark:")
print(df_landmark_final.head(10))
print(df_landmark_final.shape)

landmark_category:
                theme                                        sub_theme  \
0  Leisure/Recreation  Informal Outdoor Facility (Park/Garden/Reserve)   
1   Place Of Assembly                                          Library   

   category_id  
0            1  
1            2  

landmark:
   landmark_id                        feature_name  category_id   latitude  \
0            1  North Melbourne Recreation Reserve            1 -37.798835   
1            2                        Princes Park            1 -37.787016   
2            3                       Argyle Square            1 -37.803148   
3            4                        Kings Domain            1 -37.825524   
4            5                   Federation Square            1 -37.817852   
5            6                   Flagstaff Gardens            1 -37.811122   
6            7                       Westgate Park            1 -37.831492   
7            8                   Newmarket Reserve            1 -37.7878

In [53]:
df_landmark_category.to_csv("data/cleaned/landmark_category.csv", index=False)
df_landmark_final.to_csv("data/cleaned/landmark.csv", index=False)
print("landmark_category saved:", len(df_landmark_category), "rows")
print("landmark saved:", len(df_landmark_final), "rows")

landmark_category saved: 2 rows
landmark saved: 38 rows
